# 📱 SMS Spam & Scam Detector

## 🎯 Project Overview

This project is a **Machine Learning-based SMS Spam & Scam Detector**.

The goal was to build a model that can read an SMS message and classify it into one of two categories:

- `ham` → NOT SPAM / legitimate message
- `spam` → SPAM / SCAM message

The project follows a complete Machine Learning workflow, starting from raw text data and ending with a trained classification model that can make predictions on unseen SMS messages.

A machine learning project that classifies SMS messages as:

- **SPAM / SCAM**
- **HAM / NOT SPAM**

## Project Workflow

Dataset → Data Cleaning → EDA → Text Preprocessing → TF-IDF → Model Training → Evaluation → Real SMS Prediction

---


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully!")

Libraries imported successfully!


# labeled training dataset


In [18]:
import pandas as pd

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "text"]
)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nLabel distribution:")
print(df["label"].value_counts())

df.head()

Shape: (5572, 2)

Columns: ['label', 'text']

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [19]:


print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Missing values:
label    0
text     0
dtype: int64

Duplicate rows:
403

Data types:
label    str
text     str
dtype: object


In [20]:
df = df.drop_duplicates().reset_index(drop=True)

print("New shape:", df.shape)
print("Duplicates remaining:", df.duplicated().sum())

New shape: (5169, 2)
Duplicates remaining: 0


In [21]:
print(df["label"].unique())

<StringArray>
['ham', 'spam']
Length: 2, dtype: str


In [23]:
print(df["label"].value_counts(normalize=True) * 100)

label
ham     87.366996
spam    12.633004
Name: proportion, dtype: float64


In [ ]:
print(df["label"].unique())

<StringArray>
['ham', 'spam']
Length: 2, dtype: str


## Train-Test Split

We split the dataset into two parts:

- **Training set (80%)** → used to teach the machine learning model.
- **Testing set (20%)** → kept unseen during training and used to evaluate the model.

We use `stratify=y` so that both sets maintain approximately the same HAM/SPAM ratio as the original dataset.

We also use `random_state=42` so that we get the same split every time we run the notebook.

In [24]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting label distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training samples: 4135
Testing samples: 1034

Training label distribution:
label
ham     87.376058
spam    12.623942
Name: proportion, dtype: float64

Testing label distribution:
label
ham     87.330754
spam    12.669246
Name: proportion, dtype: float64


## Text Vectorization with TF-IDF

Machine learning models cannot directly understand raw SMS text.

TF-IDF converts each SMS message into numerical features based on the words it contains.

We will:
- Fit the TF-IDF vectorizer only on the training data.
- Transform the training data into numerical features.
- Transform the testing data using the same vocabulary.

We do NOT fit TF-IDF on the test data because the test set must remain unseen during training.

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (4135, 7680)
Testing TF-IDF shape: (1034, 7680)


## Train the Logistic Regression Model

Now that the SMS messages have been converted into numerical TF-IDF features,
we can train a machine learning classifier.

We will use Logistic Regression to learn the relationship between
the TF-IDF features and the SMS labels:

- ham
- spam

The model will learn only from the training data.

In [26]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("Model trained successfully!")

Model trained successfully!


## Make Predictions on the Test Set

The model has learned from the training data.

Now we give it the unseen test messages and ask it to predict
whether each SMS is:

- ham
- spam

In [27]:
y_pred = model.predict(X_test_tfidf)

print("Predictions made successfully!")
print("Number of predictions:", len(y_pred))

Predictions made successfully!
Number of predictions: 1034


## Model Accuracy

We compare the model's predictions (`y_pred`)
with the actual labels (`y_test`).

Accuracy tells us the percentage of test SMS messages
that the model classified correctly.

In [28]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 0.9642
Accuracy: 96.42%


## Classification Report

Accuracy gives us the overall percentage of correct predictions.

However, spam detection needs more detailed metrics.

We will calculate:

- **Precision** → How trustworthy are SPAM predictions?
- **Recall** → How much actual SPAM did we detect?
- **F1-score** → Balance between precision and recall.

In [29]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       903
        spam       0.96      0.75      0.84       131

    accuracy                           0.96      1034
   macro avg       0.96      0.87      0.91      1034
weighted avg       0.96      0.96      0.96      1034



## Confusion Matrix

A confusion matrix shows exactly how many messages
the model classified correctly and incorrectly.

It allows us to see:

- HAM predicted as HAM
- HAM predicted as SPAM
- SPAM predicted as HAM
- SPAM predicted as SPAM

This helps us understand the types of mistakes
our spam detector is making.

In [30]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[899   4]
 [ 33  98]]


### Find the Mistakes

In [31]:
errors = pd.DataFrame({
    "text": X_test,
    "actual": y_test,
    "predicted": y_pred
})

missed_spam = errors[
    (errors["actual"] == "spam") &
    (errors["predicted"] == "ham")
]

print("Missed spam messages:", len(missed_spam))

missed_spam

Missed spam messages: 33


,text,actual,predicted
5137,Want explicit SEX in 30 secs? Ring 02073162414...,spam,ham
521,You will recieve your tone within the next 24h...,spam,ham
2697,Send a logo 2 ur lover - 2 names joined by a h...,spam,ham
484,Congrats! 1 year special cinema pass for 2 is ...,spam,ham
1611,Hi if ur lookin 4 saucy daytime fun wiv busty ...,spam,ham
4752,TheMob>Hit the link to get a premium Pink Pant...,spam,ham
1405,As a registered optin subscriber ur draw 4 £10...,spam,ham
5139,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...,spam,ham
4365,Customer service announcement. We recently tri...,spam,ham
1645,"Free msg. Sorry, a service you ordered from 81...",spam,ham


# 🧪 Testing the Model on New SMS Messages

The model has already been trained and evaluated on the test dataset.

Now we will test it on completely new SMS messages that were not part of the dataset.

This helps us understand how the model behaves in a real-world situation.

In [32]:
def predict_sms(message):
    # Convert the message into TF-IDF features
    message_tfidf = tfidf.transform([message])
    
    # Make prediction
    prediction = model.predict(message_tfidf)[0]
    
    return prediction

## 📱 Test Custom SMS Messages

Let's give the trained model some new messages and see whether it predicts:

- ham → legitimate message
- spam → unwanted/promotional/scam message

In [33]:
sms1 = "Hey, are we still meeting for lunch today?"
sms2 = "Congratulations! You have won £1000. Call now to claim your prize!"

print("SMS 1:", predict_sms(sms1))
print("SMS 2:", predict_sms(sms2))

SMS 1: ham
SMS 2: spam


In [34]:
messages = [
    "Hey bro, I will reach home in 20 minutes.",
    "URGENT! You have won a cash prize. Call this number now!",
    "Can you send me the notes from today's class?",
    "FREE entry! Text WIN to 80000 to claim your reward!",
    "Mom asked me to buy some groceries on my way home.",
    "Congratulations! You have been selected for a £5000 reward."
]

for message in messages:
    print(f"{predict_sms(message):5} → {message}")

ham   → Hey bro, I will reach home in 20 minutes.
spam  → URGENT! You have won a cash prize. Call this number now!
ham   → Can you send me the notes from today's class?
spam  → FREE entry! Text WIN to 80000 to claim your reward!
ham   → Mom asked me to buy some groceries on my way home.
ham   → Congratulations! You have been selected for a £5000 reward.


## Model 2 — Multinomial Naive Bayes

We already trained Logistic Regression as our first model.

Now we will train Multinomial Naive Bayes using the same TF-IDF features.

This allows us to compare different machine learning algorithms
on exactly the same dataset and features.

In [35]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

print("Naive Bayes model trained successfully!")

Naive Bayes model trained successfully!


In [36]:
y_pred_nb = nb_model.predict(X_test_tfidf)

print("Predictions made successfully!")
print("Number of predictions:", len(y_pred_nb))

Predictions made successfully!
Number of predictions: 1034


In [37]:
from sklearn.metrics import accuracy_score

accuracy_nb = accuracy_score(y_test, y_pred_nb)

print(f"Naive Bayes Accuracy: {accuracy_nb:.4f}")
print(f"Naive Bayes Accuracy: {accuracy_nb * 100:.2f}%")

Naive Bayes Accuracy: 0.9526
Naive Bayes Accuracy: 95.26%


In [38]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       903
        spam       1.00      0.63      0.77       131

    accuracy                           0.95      1034
   macro avg       0.97      0.81      0.87      1034
weighted avg       0.96      0.95      0.95      1034



In [40]:
import joblib
from sklearn.pipeline import Pipeline

# Combine the already-trained TF-IDF and Logistic Regression
sms_spam_model = Pipeline([
    ("tfidf", tfidf),
    ("model", model)
])

# Save the complete model
joblib.dump(sms_spam_model, "sms_spam_model.pkl")

print("Final SMS Spam model saved successfully!")

Final SMS Spam model saved successfully!


* so next time ill use this code only 


In [41]:
import joblib

sms_spam_model = joblib.load("sms_spam_model.pkl")

In [42]:
sms_spam_model.predict(["Congratulations! You won a cash prize!"])

array(['spam'], dtype=object)

# 📱 SMS Spam & Scam Detector

## 🎯 Project Overview

This project is a **Machine Learning-based SMS Spam & Scam Detector**.

The goal was to build a model that can read an SMS message and classify it into one of two categories:

- `ham` → NOT SPAM / legitimate message
- `spam` → SPAM / SCAM message

The project follows a complete Machine Learning workflow, starting from raw text data and ending with a trained classification model that can make predictions on unseen SMS messages.

---

# 🗂️ Dataset

The project uses the **SMS Spam Collection** dataset.

The dataset contains SMS messages labeled as either `ham` or `spam`.

### Initial Dataset

- Total messages: **5,572**
- Columns: `label`, `text`

### Label Distribution

| Label | Percentage |
|---|---:|
| HAM | 87.37% |
| SPAM | 12.63% |

The dataset is naturally imbalanced because normal SMS messages are much more common than spam messages.

We did not artificially balance the dataset. Instead, we preserved the original distribution and used **stratified train/test splitting**.

---

# 🧹 1. Data Cleaning

Before training the model, we performed basic data-quality checks.

We checked:

- Missing values
- Duplicate rows
- Data types
- Unique labels
- Class distribution

### Missing Values

There were **no missing values** in either column.

### Duplicate Rows

The dataset contained **403 duplicate rows**.

We removed them using:

```python
df = df.drop_duplicates().reset_index(drop=True)